# Step Detection
**Pipeline:** DB laden → Norm berechnen → Glätten → Stationarität → ACF/PACF → Peak Detection → Steps-Tabelle schreiben

Orientiert an der Analyse-Pipeline aus `01_TimeSeries.pdf` (Folie 6.2 Workflow).

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

print('Imports OK ✓')

## Schritt 1 — Daten aus DB laden

In [ ]:
DB_PATH = '../data/emi_nav.db'
RUN_ID  = 'R1'  # ← R1, R2 oder R3

conn = sqlite3.connect(DB_PATH)

accel = pd.read_sql_query(f"""
    SELECT timestamp_ms, x, y, z
    FROM imu
    WHERE run_id = '{RUN_ID}' AND sensor = 'accel'
    ORDER BY timestamp_ms
""", conn)

conn.close()
print(f'Run {RUN_ID}: {len(accel)} Accelerometer-Samples geladen')
accel.head()

## Schritt 2 — Norm berechnen & Rohdaten visualisieren

In [ ]:
accel['norm'] = np.sqrt(accel['x']**2 + accel['y']**2 + accel['z']**2)
accel['time_s'] = (accel['timestamp_ms'] - accel['timestamp_ms'].min()) / 1000

plt.figure(figsize=(14, 4))
plt.plot(accel['time_s'], accel['norm'], alpha=0.7, label='||a|| roh')
plt.title(f'[{RUN_ID}] Accelerometer Magnitude (roh)')
plt.xlabel('Zeit [s]')
plt.ylabel('||a|| [m/s²]')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Schritt 3 — Sampling Rate schätzen & Low-Pass Filter

In [ ]:
dt_ms = accel['timestamp_ms'].diff().median()
fs = 1000 / dt_ms
print(f'Geschätzte Sampling Rate: {fs:.1f} Hz')

def butter_lowpass(data, cutoff=5.0, fs=250, order=4):
    nyq = fs / 2
    b, a = butter(order, cutoff / nyq, btype='low')
    return filtfilt(b, a, data)

accel['norm_filtered'] = butter_lowpass(accel['norm'], cutoff=5.0, fs=fs)

plt.figure(figsize=(14, 4))
plt.plot(accel['time_s'], accel['norm'], alpha=0.3, label='roh')
plt.plot(accel['time_s'], accel['norm_filtered'], label='gefiltert (5 Hz)', linewidth=2)
plt.title(f'[{RUN_ID}] Gefilterte Magnitude')
plt.xlabel('Zeit [s]')
plt.ylabel('||a|| [m/s²]')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Schritt 4 — Stationarität prüfen (ADF + KPSS)

Laut Analyse-Pipeline (Folie 6.2): erst Stationarität sicherstellen, dann ACF/PACF.

In [ ]:
signal = accel['norm_filtered'].dropna()

adf_stat, adf_p, *_ = adfuller(signal)
kpss_stat, kpss_p, *_ = kpss(signal, regression='c')

print(f"ADF  → p={adf_p:.4f}  {'✓ stationär' if adf_p < 0.05 else '✗ NICHT stationär → differenzieren!'}")
print(f"KPSS → p={kpss_p:.4f}  {'✓ stationär' if kpss_p > 0.05 else '✗ NICHT stationär → differenzieren!'}")

if adf_p >= 0.05 or kpss_p <= 0.05:
    accel['signal_for_detection'] = accel['norm_filtered'].diff()
    print('→ 1st-order differencing angewendet')
else:
    accel['signal_for_detection'] = accel['norm_filtered']
    print('→ Signal direkt verwendbar')

## Schritt 5 — ACF & PACF

Accelerometer zeigt typischerweise MA-Charakter (kurze Impulse = Schritte).

In [ ]:
sig = accel['signal_for_detection'].dropna()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7))
plot_acf(sig, lags=60, ax=ax1, title=f'[{RUN_ID}] ACF – Accelerometer Magnitude')
plot_pacf(sig, lags=60, ax=ax2, title=f'[{RUN_ID}] PACF – Accelerometer Magnitude')
plt.tight_layout()
plt.show()

## Schritt 6 — Peak Detection (Step Counting)

In [ ]:
sig_values = accel['signal_for_detection'].dropna().values

# Dynamischer Threshold: Mittelwert + 0.5 * Std
threshold = sig_values.mean() + 0.5 * sig_values.std()

# Mindestabstand zwischen Schritten: 300ms
min_dist = int(0.3 * fs)

peaks, _ = find_peaks(sig_values, height=threshold, distance=min_dist)

sig_times = accel['time_s'].dropna().values
step_times = sig_times[peaks]

print(f'[{RUN_ID}] Detektierte Schritte: {len(peaks)}')
print(f'Durchschnittliche Schrittfrequenz: {len(peaks) / accel["time_s"].max():.2f} Schritte/s')

plt.figure(figsize=(14, 5))
plt.plot(sig_times, sig_values, label='Signal', alpha=0.8)
plt.plot(step_times, sig_values[peaks], 'x', color='red', markersize=10, label=f'Schritte (n={len(peaks)})')
plt.axhline(threshold, color='orange', linestyle='--', label='Threshold')
plt.title(f'[{RUN_ID}] Step Detection')
plt.xlabel('Zeit [s]')
plt.ylabel('||a|| [m/s²]')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Schritt 7 — `steps` Tabelle erstellen & befüllen

Compass Direction wird aus `mag`-Daten berechnet: `atan2(y, x)` → Azimut in Grad [0°, 360°).
Nearest-neighbour Join via `merge_asof` zwischen Step-Timestamps und Magnetometer-Timestamps.

In [ ]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# Tabelle anlegen (idempotent)
cur.executescript("""
CREATE TABLE IF NOT EXISTS steps (
    id              INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id          TEXT,
    timestamp_ms    INTEGER,
    accel_norm      REAL,
    compass_deg     REAL,
    FOREIGN KEY (run_id) REFERENCES runs(run_id)
);
""")
conn.commit()
print("Tabelle 'steps' bereit ✓")

# Magnetometer laden
mag = pd.read_sql_query(f"""
    SELECT timestamp_ms, x, y
    FROM imu
    WHERE run_id = '{RUN_ID}' AND sensor = 'mag'
    ORDER BY timestamp_ms
""", conn)

# Compass Direction berechnen: 0°=Nord, 90°=Ost, 180°=Süd, 270°=West
mag['compass_deg'] = (np.degrees(np.arctan2(mag['y'], mag['x'])) + 360) % 360

# Step-DataFrame bauen
step_timestamps = accel['timestamp_ms'].dropna().values[peaks]
step_norms      = accel['norm_filtered'].dropna().values[peaks]

step_df = pd.DataFrame({
    'run_id':       RUN_ID,
    'timestamp_ms': step_timestamps,
    'accel_norm':   step_norms
})

# Nearest-neighbour Join mit Magnetometer-Daten
step_df = pd.merge_asof(
    step_df.sort_values('timestamp_ms'),
    mag[['timestamp_ms', 'compass_deg']].sort_values('timestamp_ms'),
    on='timestamp_ms',
    direction='nearest'
)

# Alte Einträge für diesen Run löschen (idempotent beim Re-Run)
cur.execute(f"DELETE FROM steps WHERE run_id = '{RUN_ID}'")
conn.commit()

# In DB schreiben
step_df[['run_id', 'timestamp_ms', 'accel_norm', 'compass_deg']].to_sql(
    'steps', conn, if_exists='append', index=False
)
conn.close()

print(f'[{RUN_ID}] {len(step_df)} Schritte in steps-Tabelle geschrieben ✓')
step_df.head(10)

## Ergebnis-Schema

```
steps
├── id            INTEGER PK AUTOINCREMENT
├── run_id        TEXT FK → runs
├── timestamp_ms  INTEGER   ← Zeitpunkt des Schritts
├── accel_norm    REAL      ← ||a|| zum Schrittzeitpunkt (m/s²)
└── compass_deg   REAL      ← Blickrichtung in Grad [0°, 360°)
```

**Tipp:** `RUN_ID` oben in Zelle 1 auf `R2` oder `R3` ändern und komplett neu durchlaufen lassen — die `DELETE`-Logik verhindert Duplikate.